# 04 — Gold Hero Marts (5 narrative tables)

Builds the five aggregate tables that drive the Medium / LinkedIn /
Power BI story:

1. `gold_hero_geo_arbitrage`        — (state × ruca_bucket) Pymt − Stdzd geo premium.
2. `gold_hero_non_par_premium`      — (specialty × is_participating) patient-exposure delta.
3. `gold_hero_jcode_concentration`  — (npi) drug-revenue share + Lorenz support.
4. `gold_hero_credentials_markup`   — (provider_tier × hcpcs_cd) chargemaster markup.
5. `gold_hero_site_neutral`         — (hcpcs_cd) Facility vs Office payment delta.

Every mart carries at least one of (`npi`, `hcpcs_cd`, `state_abrvtn`) so
Power BI's data-model relationships fan a single slicer across all 5 pages.
Each mart joins `dim_hcpcs` for HCPCS descriptions. All writes are
strict `mode("overwrite")`.

In [0]:
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import DecimalType

from utils.secrets import configure_adls_oauth
from utils.paths   import GOLD

spark = SparkSession.builder.appName("gold_hero_marts").getOrCreate()
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
configure_adls_oauth(spark, dbutils)
DECIMAL_MONEY = DecimalType(18, 2)
DECIMAL_RATIO = DecimalType(12, 6)

fact = spark.read.format("delta").load(GOLD("fact_provider_service")).cache()
fact.count()  # materialize the cache once — 5 marts reuse it

dim_hcpcs = F.broadcast(
    spark.read.format("delta").load(GOLD("dim_hcpcs"))
         .select("hcpcs_cd", "hcpcs_desc", "is_drug", "hcpcs_family")
)

## Hero #1 — `gold_hero_geo_arbitrage`
Grain: (state × ruca_bucket). Fixes the HCPCS basket to the top 50 codes
by national volume so service-mix doesn't confound state comparisons.



In [0]:
TOP_50_HCPCS = [
    r["hcpcs_cd"]
    for r in (
        fact.groupBy("hcpcs_cd").agg(F.sum("Tot_Srvcs").alias("v"))
            .orderBy(F.col("v").desc())
            .limit(50)
            .collect()
    )
]

geo_arb = (
    fact.filter(F.col("hcpcs_cd").isin(TOP_50_HCPCS))
        .filter(F.col("Avg_Mdcr_Stdzd_Amt").isNotNull())
        .groupBy("state_abrvtn", "ruca_bucket")
        .agg(
            F.sum("Geo_Premium").cast(DECIMAL_MONEY).alias("total_geo_premium"),
            F.sum("Tot_Mdcr_Pymt_Amt").cast(DECIMAL_MONEY).alias("total_pymt"),
            F.sum("Tot_Mdcr_Stdzd_Amt").cast(DECIMAL_MONEY).alias("total_stdzd"),
            F.sum("Tot_Benes").alias("total_benes_proxy"),
            F.sum("Tot_Srvcs").alias("total_services"),
            F.avg("Geo_Adj_Factor").alias("avg_geo_adj_factor"),
        )
        .withColumn(
            "geo_premium_per_bene",
            F.when(
                F.col("total_benes_proxy") > 0,
                (F.col("total_geo_premium") / F.col("total_benes_proxy")).cast(DECIMAL_MONEY),
            ),
        )
)

(geo_arb.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("state_abrvtn")
    .save(GOLD("gold_hero_geo_arbitrage")))
spark.sql(f"OPTIMIZE delta.`{GOLD('gold_hero_geo_arbitrage')}` ZORDER BY (ruca_bucket)")
print(f"gold_hero_geo_arbitrage written → {GOLD('gold_hero_geo_arbitrage')}")

gold_hero_geo_arbitrage written → abfss://gold@sthealthcareplatdev.dfs.core.windows.net/gold_hero_geo_arbitrage/


## Hero #2 — `gold_hero_non_par_premium`
Grain: (specialty) after a Y/N pivot. Weighted-average patient exposure
by `Tot_Benes`. Result is one row per specialty with both Y and N columns
side-by-side — ready for a slope chart.



In [0]:
non_par = (
    fact.groupBy("specialty", "is_participating")
        .agg(
            F.sum(F.col("Avg_Sbmtd_Chrg")    * F.col("Tot_Benes"))
              .cast(DECIMAL_MONEY).alias("wsum_sbmtd"),
            F.sum(F.col("Avg_Mdcr_Pymt_Amt") * F.col("Tot_Benes"))
              .cast(DECIMAL_MONEY).alias("wsum_pymt"),
            F.sum("Tot_Benes").alias("total_benes"),
            F.sum("Tot_Srvcs").alias("total_services"),
            F.sum("Tot_Mdcr_Pymt_Amt").cast(DECIMAL_MONEY).alias("total_mdcr_pymt"),
            F.countDistinct("npi").alias("n_providers"),
        )
        .withColumn(
            "wavg_sbmtd_chrg",
            F.when(F.col("total_benes") > 0,
                   (F.col("wsum_sbmtd") / F.col("total_benes")).cast(DECIMAL_MONEY)),
        )
        .withColumn(
            "wavg_mdcr_pymt",
            F.when(F.col("total_benes") > 0,
                   (F.col("wsum_pymt") / F.col("total_benes")).cast(DECIMAL_MONEY)),
        )
        .withColumn(
            "wavg_patient_exposure",
            (F.col("wavg_sbmtd_chrg") - F.col("wavg_mdcr_pymt")).cast(DECIMAL_MONEY),
        )
        .drop("wsum_sbmtd", "wsum_pymt")
)

non_par_pivot = (
    non_par.groupBy("specialty")
           .pivot("is_participating", [True, False])
           .agg(
               F.first("wavg_patient_exposure").alias("exposure"),
               F.first("n_providers").alias("n_providers"),
               F.first("total_mdcr_pymt").alias("total_mdcr_pymt"),
           )
           .withColumnRenamed("true_exposure",         "exposure_par_Y")
           .withColumnRenamed("false_exposure",        "exposure_par_N")
           .withColumnRenamed("true_n_providers",      "n_providers_Y")
           .withColumnRenamed("false_n_providers",     "n_providers_N")
           .withColumnRenamed("true_total_mdcr_pymt",  "total_mdcr_pymt_Y")
           .withColumnRenamed("false_total_mdcr_pymt", "total_mdcr_pymt_N")
           .withColumn(
               "non_par_premium_pct",
               F.when(
                   F.col("exposure_par_Y") > 0,
                   ((F.col("exposure_par_N") - F.col("exposure_par_Y"))
                    / F.col("exposure_par_Y")).cast(DECIMAL_RATIO),
               ),
           )
)

(non_par_pivot.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD("gold_hero_non_par_premium")))
print(f"gold_hero_non_par_premium written → {GOLD('gold_hero_non_par_premium')}")

gold_hero_non_par_premium written → abfss://gold@sthealthcareplatdev.dfs.core.windows.net/gold_hero_non_par_premium/


## Hero #3 — `gold_hero_jcode_concentration`
Grain: (npi). Per-provider drug revenue share + Lorenz-curve support columns
(`cum_share_drug`, `cum_share_providers`) so a Power BI Deneb/Vega visual
can render the curve directly without further aggregation.



In [0]:
provider_drug = (
    fact.groupBy("npi", "specialty", "state_abrvtn")
        .agg(
            F.sum("Tot_Mdcr_Pymt_Amt").cast(DECIMAL_MONEY).alias("total_mdcr_pymt"),
            F.sum(F.when(F.col("is_drug"), F.col("Tot_Mdcr_Pymt_Amt")).otherwise(0))
              .cast(DECIMAL_MONEY).alias("drug_mdcr_pymt"),
            F.sum(F.when(F.col("is_drug"), F.col("Tot_Srvcs")).otherwise(0))
              .alias("drug_services"),
            F.sum("Tot_Srvcs").alias("total_services"),
            F.countDistinct(F.when(F.col("is_drug"), F.col("hcpcs_cd"))).alias("n_distinct_jcodes"),
        )
        .withColumn(
            "drug_share",
            F.when(F.col("total_mdcr_pymt") > 0,
                   (F.col("drug_mdcr_pymt") / F.col("total_mdcr_pymt")).cast(DECIMAL_RATIO)),
        )
        .withColumn("is_drug_dependent", F.col("drug_share") > F.lit(0.60))
)

w_all = Window.orderBy(F.col("drug_mdcr_pymt").asc_nulls_first())

lorenz = (
    provider_drug.filter(F.col("drug_mdcr_pymt") > 0)
                 .withColumn("rn",             F.row_number().over(w_all))
                 .withColumn("cum_drug_pymt",  F.sum("drug_mdcr_pymt").over(w_all))
)

totals = lorenz.agg(
    F.sum("drug_mdcr_pymt").alias("total_drug"),
    F.count("*").alias("total_provs"),
).first()
total_drug  = float(totals["total_drug"] or 0)
total_provs = int(totals["total_provs"] or 1)

lorenz = (
    lorenz
    .withColumn(
        "cum_share_drug",
        (F.col("cum_drug_pymt") / F.lit(total_drug)).cast(DECIMAL_RATIO)
            if total_drug > 0 else F.lit(None).cast(DECIMAL_RATIO),
    )
    .withColumn(
        "cum_share_providers",
        (F.col("rn").cast("double") / F.lit(total_provs)).cast(DECIMAL_RATIO),
    )
    .select(
        "npi", "specialty", "state_abrvtn",
        "drug_mdcr_pymt", "drug_services", "total_mdcr_pymt",
        "drug_share", "is_drug_dependent",
        "cum_share_providers", "cum_share_drug",
    )
)

(lorenz.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD("gold_hero_jcode_concentration")))
print(f"gold_hero_jcode_concentration written → {GOLD('gold_hero_jcode_concentration')}")

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


gold_hero_jcode_concentration written → abfss://gold@sthealthcareplatdev.dfs.core.windows.net/gold_hero_jcode_concentration/


## Hero #4 — `gold_hero_credentials_markup`
Grain: (provider_tier × hcpcs_cd) pivoted across the 4 known tiers.
`Other/Unknown` is filtered out so the markup comparison stays signal-rich.



In [0]:
SHARED_E_AND_M = ["99213", "99214", "99203", "99204", "99212", "99215"]

cred_markup = (
    fact.filter(F.col("Markup_Ratio").isNotNull() & (F.col("Tot_Srvcs") >= 25))
        .filter(F.col("provider_tier") != F.lit("Other/Unknown"))
        .groupBy("provider_tier", "hcpcs_cd")
        .agg(
            F.sum(F.col("Avg_Sbmtd_Chrg")     * F.col("Tot_Srvcs"))
              .cast(DECIMAL_MONEY).alias("w_sbmtd"),
            F.sum(F.col("Avg_Mdcr_Alowd_Amt") * F.col("Tot_Srvcs"))
              .cast(DECIMAL_MONEY).alias("w_alowd"),
            F.sum(F.col("Avg_Mdcr_Pymt_Amt")  * F.col("Tot_Srvcs"))
              .cast(DECIMAL_MONEY).alias("w_pymt"),
            F.sum("Tot_Srvcs").alias("total_services"),
            F.countDistinct("npi").alias("n_providers"),
        )
        .withColumn(
            "wavg_sbmtd_chrg",
            F.when(F.col("total_services") > 0,
                   (F.col("w_sbmtd") / F.col("total_services")).cast(DECIMAL_MONEY)),
        )
        .withColumn(
            "wavg_alowd",
            F.when(F.col("total_services") > 0,
                   (F.col("w_alowd") / F.col("total_services")).cast(DECIMAL_MONEY)),
        )
        .withColumn(
            "wavg_pymt",
            F.when(F.col("total_services") > 0,
                   (F.col("w_pymt") / F.col("total_services")).cast(DECIMAL_MONEY)),
        )
        .withColumn(
            "wavg_markup_ratio",
            F.when(F.col("wavg_alowd") > 0,
                   (F.col("wavg_sbmtd_chrg") / F.col("wavg_alowd")).cast(DECIMAL_RATIO)),
        )
        .withColumn("is_shared_e_and_m", F.col("hcpcs_cd").isin(SHARED_E_AND_M))
        .drop("w_sbmtd", "w_alowd", "w_pymt")
)

cred_pivot = (
    cred_markup.groupBy("hcpcs_cd", "is_shared_e_and_m")
               .pivot("provider_tier", [
                   "Physician (MD/DO)",
                   "Nurse Practitioner",
                   "Physician Assistant",
                   "Specialist",
               ])
               .agg(
                   F.first("wavg_markup_ratio").alias("markup"),
                   F.first("wavg_pymt").alias("pymt"),
                   F.first("total_services").alias("svcs"),
               )
               .join(
                   dim_hcpcs.select("hcpcs_cd", "hcpcs_desc"),
                   "hcpcs_cd",
                   "left",
               )
)

# Sanitize column names — remove characters invalid for Delta
for col_name in cred_pivot.columns:
    clean = col_name.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")
    if clean != col_name:
        cred_pivot = cred_pivot.withColumnRenamed(col_name, clean)

(cred_pivot.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD("gold_hero_credentials_markup")))
spark.sql(f"OPTIMIZE delta.`{GOLD('gold_hero_credentials_markup')}` ZORDER BY (hcpcs_cd)")
print(f"gold_hero_credentials_markup written → {GOLD('gold_hero_credentials_markup')}")

gold_hero_credentials_markup written → abfss://gold@sthealthcareplatdev.dfs.core.windows.net/gold_hero_credentials_markup/


## Hero #5 — `gold_hero_site_neutral`
Grain: (hcpcs_cd). Filter to codes with ≥1000 services on both F and O
sides so the comparison is robust. Compute `site_neutral_savings` as the
dollar volume that would be saved if F-billed services paid at O rates.



In [0]:
site_grain = (
    fact.filter(F.col("place_of_srvc").isin("F", "O"))
        .groupBy("hcpcs_cd", "place_of_srvc")
        .agg(
            F.sum(F.col("Avg_Mdcr_Pymt_Amt")  * F.col("Tot_Srvcs"))
              .cast(DECIMAL_MONEY).alias("w_pymt"),
            F.sum(F.col("Avg_Mdcr_Stdzd_Amt") * F.col("Tot_Srvcs"))
              .cast(DECIMAL_MONEY).alias("w_stdzd"),
            F.sum("Tot_Srvcs").alias("total_services"),
            F.sum("Tot_Mdcr_Pymt_Amt").cast(DECIMAL_MONEY).alias("total_pymt_amt"),
            F.first("specialty").alias("specialty_example"),
        )
        .withColumn(
            "wavg_pymt",
            F.when(F.col("total_services") > 0,
                   (F.col("w_pymt") / F.col("total_services")).cast(DECIMAL_MONEY)),
        )
        .withColumn(
            "wavg_stdzd",
            F.when(F.col("total_services") > 0,
                   (F.col("w_stdzd") / F.col("total_services")).cast(DECIMAL_MONEY)),
        )
        .drop("w_pymt", "w_stdzd")
)

site_neutral = (
    site_grain.groupBy("hcpcs_cd")
              .pivot("place_of_srvc", ["F", "O"])
              .agg(
                  F.first("wavg_pymt").alias("pymt"),
                  F.first("total_services").alias("svcs"),
                  F.first("total_pymt_amt").alias("total"),
              )
              .filter(F.col("F_svcs").isNotNull() & F.col("O_svcs").isNotNull())
              .filter((F.col("F_svcs") >= 1000) & (F.col("O_svcs") >= 1000))
              .withColumn(
                  "site_ratio",
                  F.when(F.col("O_pymt") > 0,
                         (F.col("F_pymt") / F.col("O_pymt")).cast(DECIMAL_RATIO)),
              )
              .withColumn(
                  "site_neutral_savings",
                  ((F.col("F_pymt") - F.col("O_pymt")) * F.col("F_svcs"))
                  .cast(DECIMAL_MONEY),
              )
              .join(
                  dim_hcpcs.select("hcpcs_cd", "hcpcs_desc", "is_drug"),
                  "hcpcs_cd",
                  "left",
              )
)

(site_neutral.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD("gold_hero_site_neutral")))
spark.sql(f"OPTIMIZE delta.`{GOLD('gold_hero_site_neutral')}` ZORDER BY (site_ratio)")
print(f"gold_hero_site_neutral written → {GOLD('gold_hero_site_neutral')}")

gold_hero_site_neutral written → abfss://gold@sthealthcareplatdev.dfs.core.windows.net/gold_hero_site_neutral/


In [0]:
fact.unpersist()
print("All 5 hero marts complete.")

All 5 hero marts complete.
